# 🎯 Logistic Regression — Supermarket Sales Dataset

## Why Not Linear Regression for Binary Outcomes?

When the outcome variable is **binary** (e.g., Member vs Normal), linear regression fails because:
- It can predict probabilities **outside `[0, 1]`** — meaningless for probabilities
- The relationship between predictors and probability is inherently **non-linear**
- The error terms violate normality and homoscedasticity assumptions

**Logistic Regression** solves this by modelling the **log-odds** of the outcome:

$$\log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 X_1 + \cdots + \beta_k X_k$$

Which maps to probability through the **sigmoid function**:

$$P(Y=1 \mid X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X_1 + \cdots)}}$$

## Topics Covered

| Part | Topic |
|------|-------|
| 1 | Sigmoid function & log-odds intuition |
| 2 | Exploratory analysis by class |
| 3 | Simple logistic regression (1 predictor) |
| 4 | Feature engineering & train/test split |
| 5 | Multiple logistic regression + odds ratios |
| 6 | Evaluation — confusion matrix, ROC, PR curve |
| 7 | Decision threshold tuning |
| 8 | Regularisation — Ridge (L2) vs Lasso (L1) |
| 9 | Feature importance |
| 10 | Calibration & predicted probability analysis |

**Target variable:** `Customer type` → `1 = Member`, `0 = Normal`

---

## 🔧 Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

# statsmodels for inferential logistic regression (p-values, CIs)
import statsmodels.api as sm

# sklearn for ML-style evaluation & regularisation
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay
)
from sklearn.calibration import calibration_curve
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13
RANDOM_STATE = 42
ALPHA = 0.05

print('Libraries loaded successfully.')
print(f'Significance level alpha = {ALPHA}')

## 📂 Load & Prepare Data

In [ ]:
df = pd.read_csv('supermarket_sales.csv')

# Binary target: 1 = Member, 0 = Normal
df['is_member'] = (df['Customer type'] == 'Member').astype(int)

print(f'Dataset shape: {df.shape}')
print()
print('Target variable distribution:')
for v, c in df['is_member'].value_counts().items():
    label = 'Member' if v == 1 else 'Normal'
    print(f'  {label} ({v}): {c} rows  ({c/len(df)*100:.1f}%)')
print()
print('Classes are nearly perfectly balanced (501 vs 499).')
print('No class-imbalance handling needed.')

df.head(3)

---
# Part 1: The Sigmoid Function & Log-Odds Intuition

### Three equivalent ways to express the same thing

| Scale | Formula | Range | Linear in X? |
|-------|---------|-------|-------------|
| **Probability** | p | (0, 1) | ❌ No |
| **Odds** | p / (1−p) | (0, ∞) | ❌ No |
| **Log-odds (logit)** | log(p / 1−p) | (−∞, +∞) | ✅ Yes |

Logistic regression is **linear regression on the log-odds scale**. The sigmoid function converts log-odds back to probabilities:

$$\sigma(z) = \frac{1}{1 + e^{-z}} \quad \text{where } z = \mathbf{X}\boldsymbol{\beta}$$

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Panel 1: Sigmoid curve ---
z = np.linspace(-8, 8, 400)
sigmoid = 1 / (1 + np.exp(-z))

axes[0].plot(z, sigmoid, 'steelblue', lw=2.5)
axes[0].axhline(0.5, color='red', linestyle='--', lw=1.8, label='Default decision boundary (p=0.5)')
axes[0].axvline(0,   color='gray', linestyle=':', lw=1.5)
axes[0].fill_between(z, sigmoid, 0.5, where=(sigmoid >= 0.5), alpha=0.12, color='steelblue', label='Predict Member')
axes[0].fill_between(z, sigmoid, 0.5, where=(sigmoid <  0.5), alpha=0.12, color='tomato',    label='Predict Normal')
# Annotate a few key points
for z_val, label in [(-4, 'z=-4\np≈0.02'), (0, 'z=0\np=0.50'), (4, 'z=4\np≈0.98')]:
    p_val = 1/(1+np.exp(-z_val))
    axes[0].scatter(z_val, p_val, color='darkblue', s=60, zorder=5)
    axes[0].annotate(label, (z_val, p_val), textcoords='offset points',
                     xytext=(10, -15), fontsize=8)
axes[0].set_title('Sigmoid (Logistic) Function\n$\\sigma(z) = 1/(1+e^{-z})$')
axes[0].set_xlabel('Log-odds  z = X$\\beta$')
axes[0].set_ylabel('Probability  P(Member)')
axes[0].set_ylim(-0.05, 1.05)
axes[0].legend(fontsize=8)

# --- Panel 2: Logit (probability → log-odds) ---
p = np.linspace(0.01, 0.99, 400)
logodds = np.log(p / (1 - p))

axes[1].plot(p, logodds, 'purple', lw=2.5)
axes[1].axhline(0, color='red', linestyle='--', lw=1.5, label='log-odds=0  →  p=0.5')
axes[1].axvline(0.5, color='gray', linestyle=':', lw=1.5)
axes[1].fill_between(p, logodds, 0, where=(p >= 0.5), alpha=0.12, color='steelblue')
axes[1].fill_between(p, logodds, 0, where=(p <  0.5), alpha=0.12, color='tomato')
axes[1].set_title('Logit Function: Probability → Log-odds\nlogit(p) = log(p / 1−p)')
axes[1].set_xlabel('Probability p')
axes[1].set_ylabel('Log-odds')
axes[1].legend(fontsize=9)

# --- Panel 3: Probability → Odds → Log-odds walkthrough ---
example_ps = np.array([0.1, 0.25, 0.5, 0.6, 0.75, 0.9])
example_odds = example_ps / (1 - example_ps)
example_lo   = np.log(example_odds)
x_pos = np.arange(len(example_ps))
bars = axes[2].bar(x_pos, example_lo, color=plt.cm.RdYlBu(example_ps), edgecolor='white', width=0.6)
axes[2].axhline(0, color='black', lw=1)
for i, (p_v, lo_v, odds_v) in enumerate(zip(example_ps, example_lo, example_odds)):
    axes[2].text(i, lo_v + (0.15 if lo_v >= 0 else -0.25),
                 f'p={p_v}\nodds={odds_v:.2f}', ha='center', fontsize=7.5)
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels([f'{int(p*100)}%' for p in example_ps])
axes[2].set_title('Probability → Log-odds Conversion\n(Shows why log-odds scale is linear-friendly)')
axes[2].set_xlabel('Probability of Membership')
axes[2].set_ylabel('Log-odds')

plt.suptitle('Part 1: Logistic Regression Core Concepts', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
# Part 2: Exploratory Analysis by Class

Before modelling, we check which features show meaningful differences between Members and Normal customers. Features that separate the classes well will be good predictors.

In [ ]:
numeric_feats = ['Unit price', 'Quantity', 'Total', 'Rating', 'gross income']
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(numeric_feats):
    members = df[df['is_member'] == 1][col]
    normals = df[df['is_member'] == 0][col]
    _, p = stats.ttest_ind(members, normals, equal_var=False)

    axes[i].hist(members, bins=25, alpha=0.60, density=True, color='steelblue',
                 label=f'Member  μ={members.mean():.1f}')
    axes[i].hist(normals, bins=25, alpha=0.60, density=True, color='tomato',
                 label=f'Normal  μ={normals.mean():.1f}')
    axes[i].axvline(members.mean(), color='steelblue', linestyle='--', lw=1.5)
    axes[i].axvline(normals.mean(), color='tomato',    linestyle='--', lw=1.5)
    star = ' ⭐ sig' if p < ALPHA else ''
    axes[i].set_title(f'{col}   (p={p:.3f}{star})')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Density')
    axes[i].legend(fontsize=8)

# Categorical breakdown for Payment
pay_ct = pd.crosstab(df['Payment'], df['is_member'], normalize='index')
pay_ct.columns = ['Normal', 'Member']
pay_ct.plot(kind='bar', ax=axes[5], color=['tomato','steelblue'], edgecolor='white', alpha=0.85)
axes[5].set_title('Payment Method: Member vs Normal Split')
axes[5].set_xlabel('Payment Method')
axes[5].set_ylabel('Proportion')
axes[5].tick_params(axis='x', rotation=20)
axes[5].legend()

plt.suptitle('Part 2: Feature Distributions — Member vs Normal\n(⭐ = significant difference by Welch t-test, α=0.05)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Chi-square tests for categorical features vs membership
cat_feats = ['Branch', 'City', 'Gender', 'Payment', 'Product line']

print('=== Chi-Square Tests: Categorical Features vs Membership ===')
print(f'{"Feature":<20} {"chi2":>8} {"p-value":>10} {"Significant?":>14}')
print('-' * 56)
for feat in cat_feats:
    ct = pd.crosstab(df[feat], df['is_member'])
    chi2, p, _, _ = stats.chi2_contingency(ct)
    sig = 'YES ⭐' if p < ALPHA else 'no'
    print(f'{feat:<20} {chi2:>8.3f} {p:>10.4f} {sig:>14}')

print()
print('Features with ⭐ are associated with membership and are good model candidates.')

In [ ]:
# Correlation of numeric features with the target
num_cols = ['Unit price','Quantity','Total','Rating','gross income','is_member']
corr = df[num_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full correlation heatmap
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[0], linewidths=0.5, vmin=-1, vmax=1)
axes[0].set_title('Correlation Matrix\n(Numeric Features + Target)')

# Bar chart: correlation with target only
target_corr = corr['is_member'].drop('is_member').sort_values()
colors_bar = ['tomato' if v < 0 else 'steelblue' for v in target_corr]
axes[1].barh(target_corr.index, target_corr.values, color=colors_bar, edgecolor='white')
axes[1].axvline(0, color='black', lw=1)
axes[1].set_title('Pearson Correlation with is_member\n(closer to ±1 = stronger linear relationship)')
axes[1].set_xlabel('Correlation')

plt.tight_layout()
plt.show()

---
# Part 3: Simple Logistic Regression (1 Predictor)

We start simple — predict membership from **Total transaction amount** alone. This lets us visualise the logistic S-curve and understand coefficient interpretation before adding complexity.

**Model:**  $\log\left(\frac{P(\text{Member})}{1-P(\text{Member})}\right) = \beta_0 + \beta_1 \times \text{Total}$

In [ ]:
X_simple = sm.add_constant(df['Total'])
y_target = df['is_member']

model_simple = sm.Logit(y_target, X_simple).fit(disp=False)
print(model_simple.summary())

In [ ]:
b0 = model_simple.params['const']
b1 = model_simple.params['Total']
or1 = np.exp(b1)
p_total = model_simple.pvalues['Total']

print('=== Simple Logistic Regression: is_member ~ Total ===')
print(f'  Intercept beta0      = {b0:.4f}')
print(f'  Slope beta1 (Total)  = {b1:.6f}')
print(f'  Odds Ratio (e^beta1) = {or1:.6f}')
print(f'  p-value (Total)      = {p_total:.4f}')
print()
direction = 'SIGNIFICANT' if p_total < ALPHA else 'NOT significant'
print(f'  Total is {direction} as a predictor of membership (alpha=0.05)')
print()
print(f'  Odds Ratio interpretation:')
print(f'  For each $1 increase in Total, the ODDS of being a Member are')
print(f'  multiplied by {or1:.5f} — a very small effect.')

# Predicted probabilities for example values
print()
print('  Example predictions:')
for total in [50, 200, 400, 800]:
    log_odds = b0 + b1 * total
    prob = 1 / (1 + np.exp(-log_odds))
    print(f'    Total = ${total:4d} → P(Member) = {prob:.4f}')

In [ ]:
x_range = np.linspace(df['Total'].min(), df['Total'].max(), 300)
log_odds_range = b0 + b1 * x_range
prob_range = 1 / (1 + np.exp(-log_odds_range))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Jittered scatter + fitted sigmoid
np.random.seed(RANDOM_STATE)
jitter = np.random.uniform(-0.035, 0.035, len(df))
axes[0].scatter(df['Total'], df['is_member'] + jitter,
                alpha=0.18, s=14, c=df['is_member'], cmap='RdYlBu')
axes[0].plot(x_range, prob_range, 'k-', lw=2.5, label='Fitted logistic curve')
axes[0].axhline(0.5, color='red', linestyle='--', lw=1.8, label='Decision boundary  p=0.5')
axes[0].set_title(f'Simple Logistic Regression: P(Member) ~ Total\np-value={p_total:.4f} — Total alone is a weak predictor')
axes[0].set_xlabel('Total Transaction ($)')
axes[0].set_ylabel('P(is Member)')
axes[0].legend(fontsize=9)

# Log-odds (shows linearity on the logit scale)
axes[1].plot(x_range, log_odds_range, 'purple', lw=2.5)
axes[1].axhline(0, color='red', linestyle='--', lw=1.5, label='Log-odds = 0  ↔  p = 0.5')
axes[1].fill_between(x_range, log_odds_range, 0,
                     where=(log_odds_range >= 0), alpha=0.12, color='steelblue')
axes[1].fill_between(x_range, log_odds_range, 0,
                     where=(log_odds_range <  0), alpha=0.12, color='tomato')
axes[1].set_title('Log-Odds vs Total\n(Linear relationship on the logit scale)')
axes[1].set_xlabel('Total Transaction ($)')
axes[1].set_ylabel('Log-Odds of Being Member')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
# Part 4: Feature Engineering & Data Preparation

Before fitting the full model we:

1. **Select features** — drop `Tax 5%`, `cogs`, `gross income` because they are deterministic functions of `Total` (perfect multicollinearity)
2. **Encode categoricals** — one-hot encoding with `drop_first=True` to avoid the dummy variable trap
3. **Stratified train/test split** — 80/20, preserving the class balance in both sets
4. **Standardise** — fit scaler on train only, then apply to both (prevents data leakage)

In [ ]:
feature_cols = ['Unit price', 'Quantity', 'Total', 'Rating',
                'Branch', 'Gender', 'Payment', 'Product line']

df_model = df[feature_cols + ['is_member']].copy()

cat_cols = ['Branch', 'Gender', 'Payment', 'Product line']
df_model = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)
# convert bool dummies to int
bool_cols = df_model.select_dtypes('bool').columns.tolist()
df_model[bool_cols] = df_model[bool_cols].astype(int)

X = df_model.drop('is_member', axis=1)
y = df_model['is_member']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Total features  : {X.shape[1]}')
print(f'Feature names   : {list(X.columns)}')
print()
print(f'Training set : {len(X_train):4d} rows | Members: {y_train.sum()} ({y_train.mean()*100:.1f}%)')
print(f'Test set     : {len(X_test):4d} rows  | Members: {y_test.sum()} ({y_test.mean()*100:.1f}%)')
print()
print('Class balance preserved in both splits by stratify=y.')

---
# Part 5: Multiple Logistic Regression + Odds Ratios

We use **statsmodels** for the inferential output (p-values, confidence intervals, pseudo-R²).

### Pseudo-R² (McFadden's R²)
$$R^2_{McF} = 1 - \frac{\ln L_{\text{model}}}{\ln L_{\text{null}}}$$

- Unlike linear R², values of **0.2–0.4** represent an excellent fit
- The null model simply predicts the majority class for every observation

### Interpreting Coefficients as Odds Ratios
$$\text{Odds Ratio} = e^{\hat{\beta}_j}$$
- **OR > 1**: predictor *increases* the odds of being a Member  
- **OR < 1**: predictor *decreases* the odds of being a Member  
- **OR = 1**: predictor has no effect

In [ ]:
X_train_sm = sm.add_constant(X_train.astype(float))
model_full = sm.Logit(y_train.astype(float), X_train_sm).fit(disp=False, maxiter=200)
print(model_full.summary())

In [ ]:
params   = model_full.params.drop('const')
conf_int = model_full.conf_int().drop('const')
pvals    = model_full.pvalues.drop('const')

or_df = pd.DataFrame({
    'Coef (beta)'     : params.round(4),
    'Odds Ratio (OR)' : np.exp(params).round(4),
    'OR 95% CI Low'   : np.exp(conf_int[0]).round(4),
    'OR 95% CI High'  : np.exp(conf_int[1]).round(4),
    'p-value'         : pvals.round(4),
    'Significant'     : pvals < ALPHA
})

print('=== Odds Ratios Table ===')
print(or_df.to_string())
print()
sig_feats = or_df[or_df['Significant']]
if len(sig_feats) > 0:
    print('Significant predictors:')
    for feat, row in sig_feats.iterrows():
        direction = 'INCREASES' if row['Odds Ratio (OR)'] > 1 else 'DECREASES'
        print(f'  {feat:<35} OR={row["Odds Ratio (OR)"]:.3f}  p={row["p-value"]:.4f}  → {direction} odds')
else:
    print('No individually significant predictors at alpha=0.05.')
    print('This is common when features are jointly significant but individually weak.')
    print(f"  Model LR test p-value = {model_full.llr_pvalue:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

or_vals = np.exp(params)
ci_lo   = np.exp(conf_int[0])
ci_hi   = np.exp(conf_int[1])
xerr    = np.array([or_vals - ci_lo, ci_hi - or_vals])

colors_or = ['tomato' if pvals[i] < ALPHA else '#bbbbbb' for i in params.index]
ax.barh(range(len(params)), or_vals.values, xerr=xerr,
        color=colors_or, edgecolor='white', capsize=4, alpha=0.85)
ax.axvline(1, color='black', linestyle='--', lw=2, label='OR = 1 (no effect)')
ax.set_yticks(range(len(params)))
ax.set_yticklabels(params.index)
ax.set_title('Odds Ratios with 95% Confidence Intervals\n(Red = significant; bars crossing OR=1 are not significant)')
ax.set_xlabel('Odds Ratio  (e^beta)')

legend_elements = [
    mpatches.Patch(facecolor='tomato',  label='Significant  p < 0.05'),
    mpatches.Patch(facecolor='#bbbbbb', label='Not significant')
]
ax.legend(handles=legend_elements, fontsize=10)
plt.tight_layout()
plt.show()

---
# Part 6: Model Evaluation

## Confusion Matrix Terminology

```
                  Predicted
               Normal   Member
Actual Normal |   TN   |  FP  |   FP = False Positive (Normal predicted as Member)
Actual Member |   FN   |  TP  |   FN = False Negative (Member predicted as Normal)
```

| Metric | Formula | What it tells you |
|--------|---------|-------------------|
| **Accuracy** | (TP+TN)/(total) | Overall % correct |
| **Precision** | TP/(TP+FP) | Of all predicted Members, how many are real Members? |
| **Recall** | TP/(TP+FN) | Of all real Members, how many did we catch? |
| **F1 Score** | 2·Prec·Rec/(Prec+Rec) | Harmonic mean — balances Precision & Recall |
| **AUC-ROC** | Area under ROC curve | Overall discriminative ability, threshold-independent |

In [ ]:
# Use sklearn for clean predict/predict_proba interface
# C=1e6 gives essentially no regularisation — comparable to statsmodels
lr_base = LogisticRegression(C=1e6, solver='lbfgs', max_iter=1000, random_state=RANDOM_STATE)
lr_base.fit(X_train_sc, y_train)

y_pred = lr_base.predict(X_test_sc)
y_prob = lr_base.predict_proba(X_test_sc)[:, 1]   # P(Member)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

accuracy    = (tp + tn) / len(y_test)
precision   = tp / (tp + fp)  if (tp + fp)  else 0
recall      = tp / (tp + fn)  if (tp + fn)  else 0
specificity = tn / (tn + fp)  if (tn + fp)  else 0
f1          = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
auc         = roc_auc_score(y_test, y_prob)
null_acc    = max(y_test.mean(), 1 - y_test.mean())

print('=== Classification Performance on Test Set ===')
print(f'  Null baseline accuracy : {null_acc:.4f}  ({null_acc*100:.1f}%)')
print(f'  Model accuracy         : {accuracy:.4f}  ({accuracy*100:.1f}%)')
print(f'  Improvement over null  : {(accuracy - null_acc)*100:+.1f} ppts')
print()
print(f'  Precision   : {precision:.4f}')
print(f'  Recall      : {recall:.4f}')
print(f'  Specificity : {specificity:.4f}')
print(f'  F1 Score    : {f1:.4f}')
print(f'  AUC-ROC     : {auc:.4f}')
print()
print(f'  Confusion matrix: TP={tp}  FP={fp}  TN={tn}  FN={fn}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Confusion matrix heatmap
disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Member'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix (default threshold = 0.5)\nAccuracy = {accuracy:.3f}  |  F1 = {f1:.3f}')

# 2. ROC Curve
fpr, tpr, roc_thresh = roc_curve(y_test, y_prob)
youden_idx = np.argmax(tpr - fpr)

axes[1].plot(fpr, tpr, 'steelblue', lw=2.5, label=f'Logistic Reg. (AUC={auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'r--', lw=1.5, label='Random classifier  AUC=0.5')
axes[1].fill_between(fpr, tpr, alpha=0.12, color='steelblue')
axes[1].scatter(fpr[youden_idx], tpr[youden_idx], color='red', s=100, zorder=5,
                label=f'Best threshold = {roc_thresh[youden_idx]:.2f}\n(Youden\'s J index)')
axes[1].set_title('ROC Curve\n(Upper-left = perfect; diagonal = random)')
axes[1].set_xlabel('False Positive Rate  (1 − Specificity)')
axes[1].set_ylabel('True Positive Rate  (Recall)')
axes[1].legend(fontsize=8)

# 3. Precision-Recall Curve
prec_c, rec_c, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
axes[2].plot(rec_c, prec_c, 'darkorange', lw=2.5, label=f'Logistic Reg.  AP={ap:.3f}')
axes[2].axhline(y_test.mean(), color='red', linestyle='--', lw=1.5,
                label=f'Random baseline = {y_test.mean():.2f}')
axes[2].set_title('Precision-Recall Curve\n(Useful when class imbalance is present)')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Member (1)']))

---
# Part 7: Decision Threshold Tuning

The default threshold of **0.5** treats false positives and false negatives equally. In practice, their costs are rarely equal.

**Business examples:**
- **Loyalty marketing** — prefer high **Recall** (don't miss real Members, even if some Normals slip in)
- **Premium benefit allocation** — prefer high **Precision** (only classify as Member when very confident)

We sweep the threshold from 0.1 to 0.9 and plot how each metric changes.

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.01)
accs, precs, recs, f1s, specs = [], [], [], [], []

for t in thresholds:
    yp = (y_prob >= t).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_test, yp, labels=[0, 1]).ravel()
    total = len(y_test)
    acc_  = (tp_ + tn_) / total
    pre_  = tp_ / (tp_ + fp_) if (tp_ + fp_) > 0 else 0
    rec_  = tp_ / (tp_ + fn_) if (tp_ + fn_) > 0 else 0
    spe_  = tn_ / (tn_ + fp_) if (tn_ + fp_) > 0 else 0
    f1_   = 2*pre_*rec_/(pre_+rec_) if (pre_+rec_) > 0 else 0
    accs.append(acc_); precs.append(pre_); recs.append(rec_)
    specs.append(spe_); f1s.append(f1_)

best_t_f1  = thresholds[np.argmax(f1s)]
best_t_acc = thresholds[np.argmax(accs)]

print(f'Threshold = 0.50 (default) → Accuracy={accuracy:.4f}  F1={f1:.4f}')
print(f'Best by F1      → threshold = {best_t_f1:.2f}  F1={max(f1s):.4f}')
print(f'Best by Accuracy→ threshold = {best_t_acc:.2f}  Acc={max(accs):.4f}')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresholds, accs,  lw=2,   color='steelblue',  label='Accuracy')
ax.plot(thresholds, precs, lw=2,   color='green',      label='Precision')
ax.plot(thresholds, recs,  lw=2,   color='tomato',     label='Recall')
ax.plot(thresholds, f1s,   lw=2.5, color='purple',     label='F1 Score', linestyle='--')
ax.plot(thresholds, specs, lw=2,   color='darkorange', label='Specificity', linestyle=':')
ax.axvline(0.5,       color='gray',   linestyle='--', lw=1.5, alpha=0.8, label='Default threshold = 0.50')
ax.axvline(best_t_f1, color='purple', linestyle='-',  lw=1.5, alpha=0.7, label=f'Best F1 threshold = {best_t_f1:.2f}')
ax.set_title('Classification Metrics vs Decision Threshold\n(Tune based on business cost of FP vs FN)')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9, loc='lower left')
plt.tight_layout()
plt.show()

In [ ]:
# Compare default vs best-F1 threshold
print('=== Threshold Comparison ===')
print(f'{"Metric":<14} {"threshold=0.50":>16} {f"threshold={best_t_f1:.2f}":>16}')
print('-' * 48)
for metric, default_val, best_val in [
    ('Accuracy',    accs[list(thresholds).index(min(thresholds, key=lambda x: abs(x - 0.50)))],  max(accs)),
    ('Precision',   precs[list(thresholds).index(min(thresholds, key=lambda x: abs(x - 0.50)))], precs[np.argmax(f1s)]),
    ('Recall',      recs[list(thresholds).index(min(thresholds, key=lambda x: abs(x - 0.50)))],  recs[np.argmax(f1s)]),
    ('F1 Score',    f1s[list(thresholds).index(min(thresholds, key=lambda x: abs(x - 0.50)))],   max(f1s)),
    ('Specificity', specs[list(thresholds).index(min(thresholds, key=lambda x: abs(x - 0.50)))], specs[np.argmax(f1s)]),
]:
    change = best_val - default_val
    print(f'{metric:<14} {default_val:>16.4f} {best_val:>16.4f}   ({change:+.4f})')

---
# Part 8: Regularised Logistic Regression — Ridge vs Lasso

**Regularisation** adds a penalty to the loss function to prevent overfitting by shrinking large coefficients:

| Type | Penalty | Key Property |
|------|---------|-------------|
| **L2 (Ridge)** | $\lambda \sum \beta_j^2$ | Shrinks all coefficients; none reach exactly 0 |
| **L1 (Lasso)** | $\lambda \sum |\beta_j|$ | Drives some coefficients to **exactly 0** → automatic feature selection |

In sklearn, `C = 1/λ`:
- **Small C** = strong regularisation  
- **Large C** = weak regularisation (closer to ordinary logistic regression)

We find the best `C` using 5-fold cross-validation.

In [ ]:
C_values = np.logspace(-3, 3, 50)
cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

l2_mean, l2_std = [], []
l1_mean, l1_std = [], []

for C in C_values:
    pipe_l2 = Pipeline([
        ('sc', StandardScaler()),
        ('lr', LogisticRegression(C=C, penalty='l2', solver='lbfgs',
                                  max_iter=1000, random_state=RANDOM_STATE))
    ])
    sc = cross_val_score(pipe_l2, X_train, y_train, cv=cv_strat, scoring='roc_auc')
    l2_mean.append(sc.mean()); l2_std.append(sc.std())

    pipe_l1 = Pipeline([
        ('sc', StandardScaler()),
        ('lr', LogisticRegression(C=C, penalty='l1', solver='liblinear',
                                  max_iter=1000, random_state=RANDOM_STATE))
    ])
    sc = cross_val_score(pipe_l1, X_train, y_train, cv=cv_strat, scoring='roc_auc')
    l1_mean.append(sc.mean()); l1_std.append(sc.std())

best_C_l2 = C_values[np.argmax(l2_mean)]
best_C_l1 = C_values[np.argmax(l1_mean)]

print(f'Best C (Ridge L2): {best_C_l2:.4f}  →  CV AUC = {max(l2_mean):.4f}')
print(f'Best C (Lasso L1): {best_C_l1:.4f}  →  CV AUC = {max(l1_mean):.4f}')

In [ ]:
l2m = np.array(l2_mean); l2e = np.array(l2_std)
l1m = np.array(l1_mean); l1e = np.array(l1_std)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# CV AUC vs C
axes[0].semilogx(C_values, l2m, 'steelblue', lw=2.5, label='L2 Ridge')
axes[0].fill_between(C_values, l2m - l2e, l2m + l2e, alpha=0.15, color='steelblue')
axes[0].semilogx(C_values, l1m, 'green', lw=2.5, label='L1 Lasso')
axes[0].fill_between(C_values, l1m - l1e, l1m + l1e, alpha=0.15, color='green')
axes[0].axvline(best_C_l2, color='steelblue', linestyle='--', lw=1.5,
                label=f'Best C L2 = {best_C_l2:.3f}')
axes[0].axvline(best_C_l1, color='green',     linestyle='--', lw=1.5,
                label=f'Best C L1 = {best_C_l1:.3f}')
axes[0].set_title('5-Fold CV AUC vs Regularisation Strength\n(Shaded = ±1 std across folds)')
axes[0].set_xlabel('C  (1/lambda)  →  lower = stronger regularisation')
axes[0].set_ylabel('CV AUC-ROC')
axes[0].legend(fontsize=9)

# Lasso coefficient path — shows which features survive
coef_path = []
for C in C_values:
    lr_l1 = LogisticRegression(C=C, penalty='l1', solver='liblinear',
                                max_iter=1000, random_state=RANDOM_STATE)
    lr_l1.fit(X_train_sc, y_train)
    coef_path.append(lr_l1.coef_[0].copy())
coef_path = np.array(coef_path)

cmap_path = plt.cm.tab20(np.linspace(0, 1, X_train.shape[1]))
for i, (feat, col) in enumerate(zip(X_train.columns, cmap_path)):
    axes[1].semilogx(C_values, coef_path[:, i], lw=1.6, color=col, label=feat)
axes[1].axhline(0, color='black', lw=1, alpha=0.4)
axes[1].axvline(best_C_l1, color='red', linestyle='--', lw=2,
                label=f'Best C = {best_C_l1:.3f}')
axes[1].set_title('Lasso Coefficient Path\n(Features driven to 0 are excluded from model)')
axes[1].set_xlabel('C  →  weaker regularisation →')
axes[1].set_ylabel('Coefficient value')
axes[1].legend(fontsize=7, loc='upper left', ncol=2, bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

---
# Part 9: Final Model Comparison & Feature Importance

We compare three variants on the held-out test set and identify the most important features from the best model.

In [ ]:
models_compare = {
    'No regularisation': LogisticRegression(C=1e6, solver='lbfgs',
                                            max_iter=1000, random_state=RANDOM_STATE),
    f'Ridge L2 (C={best_C_l2:.3f})': LogisticRegression(C=best_C_l2, penalty='l2', solver='lbfgs',
                                                          max_iter=1000, random_state=RANDOM_STATE),
    f'Lasso L1 (C={best_C_l1:.3f})': LogisticRegression(C=best_C_l1, penalty='l1', solver='liblinear',
                                                          max_iter=1000, random_state=RANDOM_STATE),
}

results_rows = []
cv_aucs = {}

for name, mdl in models_compare.items():
    mdl.fit(X_train_sc, y_train)
    yp  = mdl.predict(X_test_sc)
    ypr = mdl.predict_proba(X_test_sc)[:, 1]
    tn_, fp_, fn_, tp_ = confusion_matrix(y_test, yp).ravel()
    acc_ = (tp_+tn_)/len(y_test)
    pre_ = tp_/(tp_+fp_) if (tp_+fp_) else 0
    rec_ = tp_/(tp_+fn_) if (tp_+fn_) else 0
    f1_  = 2*pre_*rec_/(pre_+rec_) if (pre_+rec_) else 0
    auc_ = roc_auc_score(y_test, ypr)
    cv_  = cross_val_score(mdl, X_train_sc, y_train, cv=cv_strat, scoring='roc_auc').mean()
    nz_  = np.sum(mdl.coef_[0] != 0)
    cv_aucs[name] = cv_
    results_rows.append({'Model': name, 'Accuracy': acc_, 'Precision': pre_,
                         'Recall': rec_, 'F1': f1_, 'Test AUC': auc_,
                         'CV AUC': cv_, 'Non-zero coefs': nz_})

res_df = pd.DataFrame(results_rows).set_index('Model').round(4)
print('=== Test Set Performance ===')
print(res_df.to_string())

In [ ]:
best_name = max(cv_aucs, key=cv_aucs.get)
best_mdl  = models_compare[best_name]
best_mdl.fit(X_train_sc, y_train)

feat_imp = pd.Series(best_mdl.coef_[0], index=X_train.columns)
feat_imp_sorted = feat_imp.reindex(feat_imp.abs().sort_values(ascending=True).index)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Feature importance (standardised coefficients)
colors_fi = ['steelblue' if v > 0 else 'tomato' for v in feat_imp_sorted.values]
axes[0].barh(range(len(feat_imp_sorted)), feat_imp_sorted.values,
             color=colors_fi, edgecolor='white', alpha=0.85)
axes[0].set_yticks(range(len(feat_imp_sorted)))
axes[0].set_yticklabels(feat_imp_sorted.index)
axes[0].axvline(0, color='black', lw=1.5)
legend_fi = [
    mpatches.Patch(facecolor='steelblue', label='Increases P(Member)'),
    mpatches.Patch(facecolor='tomato',    label='Decreases P(Member)')
]
axes[0].legend(handles=legend_fi, fontsize=9)
axes[0].set_title(f'Feature Importance — Standardised Coefficients\nModel: {best_name}')
axes[0].set_xlabel('Standardised Coefficient (beta)')

# ROC curves for all three models
ls_list = ['-', '--', ':']
col_list = ['steelblue', 'darkorange', 'green']
for (name, mdl), ls, col in zip(models_compare.items(), ls_list, col_list):
    ypr = mdl.predict_proba(X_test_sc)[:, 1]
    fpr_, tpr_, _ = roc_curve(y_test, ypr)
    auc_ = roc_auc_score(y_test, ypr)
    axes[1].plot(fpr_, tpr_, lw=2.5, linestyle=ls, color=col,
                 label=f'{name}  (AUC={auc_:.3f})')

axes[1].plot([0, 1], [0, 1], 'r--', lw=1.5, label='Random  AUC=0.5')
axes[1].set_title('ROC Curve: All Models on Test Set')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f'Best model: {best_name}')
print(f'Non-zero features: {(feat_imp != 0).sum()} / {len(feat_imp)}')

---
# Part 10: Probability Calibration & Model Diagnostics

A **well-calibrated** model produces predicted probabilities that match actual observed rates:
- If the model assigns 70% probability to 100 customers, ~70 of them should actually be Members
- A **calibration curve (reliability diagram)** plots mean predicted probability vs observed fraction
- Points near the diagonal = good calibration

In [ ]:
y_prob_best = best_mdl.predict_proba(X_test_sc)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Calibration curve
frac_pos, mean_pred = calibration_curve(y_test, y_prob_best, n_bins=10)
axes[0].plot(mean_pred, frac_pos, 's-', color='steelblue', lw=2.5,
             markersize=8, label='Logistic Regression')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
axes[0].set_title('Calibration Curve (Reliability Diagram)\n(Points on diagonal = perfectly calibrated)')
axes[0].set_xlabel('Mean Predicted Probability')
axes[0].set_ylabel('Fraction of True Members')
axes[0].legend(fontsize=10)

# 2. Predicted probability histograms by true class
axes[1].hist(y_prob_best[y_test == 1], bins=25, alpha=0.65, density=True,
             color='steelblue', label='True Members')
axes[1].hist(y_prob_best[y_test == 0], bins=25, alpha=0.65, density=True,
             color='tomato',    label='True Normals')
axes[1].axvline(0.5, color='black', linestyle='--', lw=2, label='Threshold = 0.5')
ambiguous = np.mean((y_prob_best > 0.4) & (y_prob_best < 0.6))
axes[1].set_title(f'Predicted P(Member) by True Class\n({ambiguous*100:.1f}% of obs in ambiguous zone 0.4-0.6)')
axes[1].set_xlabel('Predicted P(Member)')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=10)

# 3. Cumulative score distribution (ECDF) — visualise separation
for label, col, cls in [('Members', 'steelblue', 1), ('Normals', 'tomato', 0)]:
    subset = np.sort(y_prob_best[y_test == cls])
    ecdf   = np.arange(1, len(subset)+1) / len(subset)
    axes[2].plot(subset, ecdf, lw=2.5, color=col, label=label)
axes[2].axvline(0.5, color='black', linestyle='--', lw=2, label='Threshold = 0.5')
axes[2].set_title('ECDF of Predicted Scores by Class\n(Greater horizontal gap = better separation)')
axes[2].set_xlabel('Predicted P(Member)')
axes[2].set_ylabel('Cumulative Proportion')
axes[2].legend(fontsize=9)

plt.suptitle(f'Part 10: Calibration & Score Diagnostics — {best_name}', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Summary stats on predicted probabilities
for cls, label in [(1, 'Members'), (0, 'Normals')]:
    probs = y_prob_best[y_test == cls]
    print(f'{label}:')
    print(f'  Mean predicted P(Member) = {probs.mean():.4f}')
    print(f'  Std                      = {probs.std():.4f}')
    print(f'  Median                   = {np.median(probs):.4f}')
    print(f'  % above threshold 0.5   = {(probs > 0.5).mean()*100:.1f}%')
    print()

---
# Summary

## Full Workflow Recap

| Part | What We Did | Key Output |
|------|------------|------------|
| 1 | Sigmoid & log-odds intuition | Conceptual foundation |
| 2 | EDA by class | Identified informative features |
| 3 | Simple logistic regression | S-curve visualisation; weak single predictor |
| 4 | Feature engineering & splits | 800 train / 200 test; stratified; standardised |
| 5 | Multiple logistic regression | Odds ratios + significance via statsmodels |
| 6 | Evaluation | Confusion matrix, ROC-AUC, PR curve |
| 7 | Threshold tuning | Optimal threshold depends on business goal |
| 8 | Regularisation | Cross-validated best C for L1 & L2 |
| 9 | Feature importance | Standardised coefficients from best model |
| 10 | Calibration | Reliability diagram; score density by class |

## Key Concepts Cheat Sheet

| Concept | Formula / Meaning |
|---------|------------------|
| **Logit** | log(p / 1−p) — log-odds, linear in predictors |
| **Sigmoid** | σ(z) = 1/(1+e⁻ᶻ) — maps log-odds → probability |
| **Coefficient β** | Change in log-odds per unit increase in X |
| **Odds Ratio e^β** | Multiplicative change in odds per unit increase in X |
| **Threshold** | Default 0.5; tune based on cost of FP vs FN |
| **AUC-ROC** | 0.5 = random, 1.0 = perfect; threshold-independent |
| **L2 Ridge** | Shrinks all β toward 0; keeps all features |
| **L1 Lasso** | Sets some β exactly to 0 → feature selection |
| **C (sklearn)** | C = 1/λ; smaller C = stronger regularisation |
| **McFadden R²** | 0.2–0.4 = excellent for logistic regression |

---
> **Next steps:** Time Series Analysis (sales trends over time), or Customer Segmentation with K-Means Clustering.